# 03 — Predictive I: baselines before models

*What will happen?* — mirroring `references/Day3_Team_Workbook.ipynb`.

A model is only worth anything against a baseline that already knows something. Here that
baseline is the six-month runway rule the audit found before any model existed.


Reusable logic belongs in `src/startup_outcomes/`, not here — this notebook imports it and
defines nothing. Strip all outputs before committing, and check that it still runs under
**Restart & Run All**.

In [ ]:
# Activates the inline backend so a returned Figure renders. plots.py itself never
# imports pyplot, so no global state is created by the library.
%matplotlib inline

from startup_outcomes import config, features, plots
from startup_outcomes.load import load_raw
from startup_outcomes.models import baselines, protocol, supervised

df = load_raw(verify_checksum=True)
analysis = features.canonical_frame(df)
design, target = features.design_matrix(analysis)
print(f"{len(analysis):,} rows after dropping the leaky stage; {design.shape[1]} features")
print(f"base rate: {target.mean() * 100:.2f}% closed")

## Step 1 — what the model is allowed to look at

Never a hand-written list. `model_features()` is the allow-list, `assert_no_leakage` is the
guard, and both come from the timing contract in `config.py`.

In [ ]:
print("features :", config.model_features())
print("excluded :", config.EXCLUDED_LEAKY)
config.assert_no_leakage(design.columns)
print("\nthe automatic leak scan finds nothing — only the timing contract catches these:")
print(features.leakage_scan(design, target))

## Step 2 — the score to beat

Three baselines in increasing order of what they know. The third is the honest one: a
single comparison against six months of runway.

In [ ]:
results = [
    protocol.evaluate(name, estimator, design, target)
    for name, estimator in baselines.all_baselines().items()
]
print(protocol.leaderboard(results).to_string())

## Step 3 — the first model, and what the numbers mean

Accuracy is not reported as a headline anywhere in this project. At a 14% positive rate it
rewards predicting "survives" forever — as the next cell shows precisely.

In [ ]:
scores = protocol.out_of_fold_probabilities(supervised.chosen_estimator(), design, target)
print(protocol.score_probabilities(target, scores))
print(protocol.fold_spread(target, scores))
print()
for key, value in protocol.threshold_counts(target, scores).items():
    print(f"  {key:<34} {value}")

Read those two accuracy figures together. The model flags 2 companies out of 24,467, so it
scores *below* the majority baseline while being a better ranker than chance. It is a
ranking tool, not a classifier.

## Step 4 — does it beat the baseline?

"Beats" is arithmetic here: the paired bootstrap interval on the difference either
excludes zero or it does not.

In [ ]:
print(supervised.beats_baseline(design, target, scores=scores))
print()
print(
    supervised.beats_baseline(design, target, baseline_name="runway threshold rule", scores=scores)
)

## Step 5 — watching a tree overfit

The most important picture of the day. An unlimited tree reaches a perfect training score
while its out-of-fold score falls back to the base rate: there is nothing beyond the one
rule to learn.

In [ ]:
sweep = supervised.depth_sweep(design, target)
print(sweep.to_string())
plots.depth_sweep_curve(sweep)

## Day 3 deliverable

- **Baseline first:** majority 0.141 PR-AUC, runway threshold rule 0.186, runway column
  alone 0.191.
- **The model:** 0.193, fold spread ±0.006.
- **The only three questions that matter:** Does it beat the baseline? *No — the interval
  on the difference contains zero.* Is it stable? *Yes, but a single split moves it more
  than the difference does.* Can you explain it? *Yes: it is the runway rule.*

## Scratch